# ConvPhysical Encoder Analysis: Particle Dynamics Experiment

Interpretability and hypothesis testing for the `ConvPhysicalEncoder` on the particle dynamics dataset.

**Dataset:** A blob of `num_blob` particles following a circular trajectory (radius 3, $\omega$=0.05) embedded among `num_noise` temporally correlated noise particles following a 2D AR(1) process ($\alpha=0.8$), distinct from the coherent blob. Observations are standardized, interleaved $(x,y)$ coordinates sorted by $x$-position at $t=0$.

**Encoder:** `ConvPhysical` bins each timestep's particle coordinates onto a 2D occupancy grid and applies symmetric `Conv2d` filters in physical space. Learned filters are directly interpretable as spatial detectors — overlaying them with the blob's trajectory path reveals whether CPIC has learned to attend to the coherent structure.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

experiment_path = Path("../experiments/particle_experiment").resolve()
src_path = Path("../src").resolve()
for p in (experiment_path, src_path):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from generate_particle_dynamics import generate_particle_process_timeseries
from filter_visualization import (
    plot_physical_filter_heatmaps,
    compute_trajectory_in_grid_coords,
    compute_avg_density_grid,
    compute_tangential_arrows,
    plot_trajectory_density_map,
    overlay_trajectory_on_ax,
)
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from cpic import CPIC
from cpic.utils.data import PastFutureDataset

print(sys.executable)

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "mps"
print("device:", device)

period_steps = int(round((2 * np.pi) / 0.05))  # ~126 timesteps per orbit
T = period_steps // 4                           # quarter-period window (~31 steps)
print(f"period_steps={period_steps}  T={T}")

conv_phys_base = {
    "deterministic": False,
    "encoder_type": "conv_physical",
    "linear_encoder": False,
    "n_layers": 0,
    "activation": "relu",
    "grid_size": 20,
    "spatial_bounds": 3.0,
    "conv_kernel_size": 3,
    "conv_stride": 1,
    "conv_padding": 1,
}

In [ ]:
import gc

def build_past_windows_and_gt(data, gt_latent, window_size, model, t_min=None, t_max=None):
    n_timesteps = len(data)
    t_min = window_size if t_min is None else max(window_size, int(t_min))
    t_max = n_timesteps if t_max is None else min(n_timesteps, int(t_max))
    end_times = np.arange(t_min, t_max)
    past_windows = np.stack([data[t - window_size : t] for t in end_times], axis=0)
    past_tensor = torch.from_numpy(past_windows).float().to(device)
    with torch.no_grad():
        z = model.encode(past_tensor)[:, -1, :].cpu().numpy()
    return z, gt_latent[end_times - 1]


def cleanup_model_resources(*objs, verbose=True):
    for obj in objs:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()
        if verbose:
            print(f"[cleanup] CUDA alloc={torch.cuda.memory_allocated()/1024**2:.1f}MB")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        if verbose: print("[cleanup] MPS backend active.")
    elif verbose:
        print("[cleanup] CPU backend active.")

## **Hypothesis:** number of blob particles controls compression ease

**Claim:** More blob particles grouped together → more "votes" for the same coherent pattern → CPIC compresses that structure more easily, even with many noise particles present.

**Design:** Vary `num_blob` $\in$ {5, 10, 20, 30, 50} with `num_noise=80` fixed. For each run:
- Train a fresh ConvPhysical (observation space) encoder.
- Evaluate a linear probe: **test $R^2$** for recovering blob centroid position.
- Compute a **filter–trajectory alignment score**: fraction of each filter's weight mass that lands on the blob's trajectory path on the grid.

If the hypothesis holds, both test $R^2$ and trajectory alignment should increase with `num_blob`.

In [ ]:
import torch.nn.functional as F

def trajectory_filter_alignment(weights, trajectory_grid, avg_density, grid_size):
    """
    Per-filter trajectory selectivity: fraction of the filter's response on the density
    map that lands on the trajectory path.

    Applies each |filter| as a spatial detector via conv2d on avg_density (20x20),
    then measures how much of the resulting response map overlaps the trajectory path mask.
    Score > average means the filter responds more strongly on the trajectory than off it.
    """
    traj_mask = np.zeros((grid_size, grid_size), dtype=np.float32)
    oi = np.clip(trajectory_grid[:, 1].astype(int), 0, grid_size - 1)
    oj = np.clip(trajectory_grid[:, 0].astype(int), 0, grid_size - 1)
    traj_mask[oi, oj] = 1.0
    traj_mask_norm = traj_mask / (traj_mask.sum() + 1e-8)

    K = weights.shape[2]
    pad = K // 2
    density_t = torch.from_numpy(avg_density).float().unsqueeze(0).unsqueeze(0)  # (1,1,H,W)

    scores = []
    for c in range(weights.shape[0]):
        kernel = torch.from_numpy(np.abs(weights[c:c+1])).float()  # (1,1,K,K)
        response = F.conv2d(density_t, kernel, padding=pad).squeeze().numpy()  # (H,W)
        response_norm = response / (response.sum() + 1e-8)
        scores.append(float(np.sum(response_norm * traj_mask_norm)))
    return np.array(scores)


def run_conv_physical_step(num_blob, num_noise, conv_phys_base, T, device,
                            t_max=2000, seed=42, predictive_space="observation",
                            beta=1e-5, trajectory="circle",
                            semi_major=None, semi_minor=None):
    """One sweep step: generate data, train ConvPhysical, probe, return metrics + artifacts."""
    d, gt, porder, plabels, smean, sstd, pos = generate_particle_process_timeseries(
        t_max=t_max, num_blob=num_blob, num_noise=num_noise,
        trajectory=trajectory, orbit_radius=3.0,
        semi_major=semi_major, semi_minor=semi_minor,
        omega=0.05, sigma_blob=0.1,
        noise_ar_coeff=0.8, spatial_bounds=10.0, seed=seed,
    )
    n_feat = d.shape[1]
    t_split = int(0.7 * len(d))
    train_ds = PastFutureDataset([d[:t_split]], window_size=T)

    cpic = CPIC(ydim=2, xdim=n_feat, T=T, encoder_params=conv_phys_base.copy(),
                hidden_dim=64, beta=beta, device=device, predictive_space=predictive_space)
    cpic.to(device)
    loss_, I_c_, I_p_ = cpic.fit(X=train_ds, epochs=80, batch_size=256, lr=2e-4, early_stop=20)

    z_tr, gt_tr = build_past_windows_and_gt(d, gt, T, cpic, t_min=T, t_max=t_split)
    z_te, gt_te = build_past_windows_and_gt(d, gt, T, cpic, t_min=t_split, t_max=len(d))
    reg = LinearRegression().fit(z_tr, gt_tr)
    r2_train = r2_score(gt_tr, reg.predict(z_tr))
    r2_test  = r2_score(gt_te, reg.predict(z_te))

    w, meta = cpic.encoder.get_filters(layer_idx=0)
    trajectory_grid = compute_trajectory_in_grid_coords(
        pos, plabels, porder, smean, sstd,
        grid_size=meta["grid_size"], spatial_bounds=meta["spatial_bounds"],
    )
    avg_density = compute_avg_density_grid(
        d[:t_split], grid_size=meta["grid_size"], spatial_bounds=meta["spatial_bounds"],
    )
    aligns = trajectory_filter_alignment(w, trajectory_grid, avg_density, meta["grid_size"])

    result = dict(
        num_blob=num_blob, num_noise=num_noise,
        r2_train=r2_train, r2_test=r2_test,
        filters=w.copy(), trajectory_grid=trajectory_grid, avg_density=avg_density,
        align_scores=aligns, max_align=float(aligns.max()), mean_align=float(aligns.mean()),
        meta=meta, trajectory=trajectory,
    )
    cleanup_model_resources(cpic, loss_, I_c_, I_p_)
    return result

In [ ]:
num_blob_values = [5, 10, 20, 30, 50]
num_noise_fixed = 80
seeds = [0, 1, 2, 3, 4]

# 5-seed sweep. `sweep_seed_records` holds per-(num_blob, seed) metrics for the
# aggregated trend plot below; `sweep_results` keeps the first-seed (seed 0) full
# result per num_blob (filters / density / align_scores) so the downstream
# filter-grid and radial-density cells keep working unchanged.
sweep_results = []
sweep_seed_records = []
for nb_val in num_blob_values:
    print()
    print(f"--- num_blob={nb_val}, num_noise={num_noise_fixed} ---")
    for si, seed in enumerate(seeds):
        res = run_conv_physical_step(
            num_blob=nb_val,
            num_noise=num_noise_fixed,
            conv_phys_base=conv_phys_base,
            T=T,
            device=device,
            predictive_space="observation",
            seed=seed,
        )
        sweep_seed_records.append(dict(
            num_blob=nb_val, seed=seed,
            r2_test=res["r2_test"],
            max_align=res["max_align"],
            mean_align=res["mean_align"],
        ))
        if si == 0:
            sweep_results.append(res)  # representative seed-0 result for filter/density viz
        print(f"  seed={seed}  R² test={res['r2_test']:.3f}  max align={res['max_align']:.4f}")

print()
print("5-seed sweep complete.")


In [ ]:
# Summary: R-squared and filter-trajectory alignment vs num_blob (5-seed mean +/- std)
nb_vals = num_blob_values
n_seeds = len({r["seed"] for r in sweep_seed_records})

def _agg(key):
    means, stds = [], []
    for nb_val in nb_vals:
        vals = [r[key] for r in sweep_seed_records if r["num_blob"] == nb_val]
        means.append(np.mean(vals)); stds.append(np.std(vals))
    return np.array(means), np.array(stds)

r2_m,  r2_s  = _agg("r2_test")
mxa_m, mxa_s = _agg("max_align")
mna_m, mna_s = _agg("mean_align")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(nb_vals, r2_m, "o-", color="steelblue", linewidth=2, markersize=8)
ax1.fill_between(nb_vals, r2_m - r2_s, r2_m + r2_s, color="steelblue", alpha=0.2)
for x, y in zip(nb_vals, r2_m):
    ax1.annotate(f"{y:.2f}", (x, y), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9)
ax1.axhline(0, color="gray", linestyle="--", linewidth=1)
ax1.set_ylim(0, 1.09)
ax1.set_xlabel('Number of "blob" particles (num_blob)')
ax1.set_ylabel("Test R²")
ax1.set_title(f"Linear Probe R² vs blob count\n(ConvPhysical obs, num_noise={num_noise_fixed}, {n_seeds} seeds)")
ax1.grid(True, alpha=0.3)

ax2.plot(nb_vals, mxa_m, "s-", color="darkorange", linewidth=2, markersize=8, label="max align")
ax2.fill_between(nb_vals, mxa_m - mxa_s, mxa_m + mxa_s, color="darkorange", alpha=0.18)
ax2.plot(nb_vals, mna_m, "^--", color="tomato", linewidth=1.5, markersize=7, label="mean align")
ax2.fill_between(nb_vals, mna_m - mna_s, mna_m + mna_s, color="tomato", alpha=0.12)
ax2.set_ylim(bottom=0)
ax2.set_xlabel('Number of "blob" particles (num_blob)')
ax2.set_ylabel("Trajectory-filter alignment score")
ax2.set_title(f"Filter-trajectory alignment vs blob count\n(higher = filter attends more to trajectory path, {n_seeds} seeds)")
ax2.legend()
ax2.grid(True, alpha=0.3)

fig.suptitle("Hypothesis: more blob particles -> easier coherent structure extraction", y=1.02)
fig.tight_layout()
out = Path("res") / "convphys_r2_align_vs_numblob_circle.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print("saved", out)
plt.show()


In [ ]:
# Filter grid: avg_density (input before conv) + top-4 trajectory-aligned response maps
# First column = raw input density; remaining columns = response after applying each filter.
n_top  = 4
n_cols = n_top + 1
n_runs = len(sweep_results)

fig, axes = plt.subplots(n_runs, n_cols, figsize=(n_cols * 2.4, n_runs * 2.4), constrained_layout=True)
if n_runs == 1:
    axes = axes[np.newaxis, :]

for row, res in enumerate(sweep_results):
    w               = res["filters"]
    trajectory_grid = res["trajectory_grid"]
    avg_density     = res["avg_density"]
    aligns          = res["align_scores"]
    top_idx         = np.argsort(aligns)[::-1][:n_top]
    arrows          = compute_tangential_arrows(trajectory_grid)
    gs              = res["meta"]["grid_size"]

    density_t = torch.from_numpy(avg_density).float().unsqueeze(0).unsqueeze(0)
    K = w.shape[2]; pad = K // 2

    # Column 0: avg_density input (before any convolution)
    ax0 = axes[row, 0]
    ax0.imshow(avg_density, cmap="magma", aspect="equal", origin="lower",
               extent=[0, gs, 0, gs])
    overlay_trajectory_on_ax(ax0, trajectory_grid, arrows=arrows)
    ax0.set_title("avg density\n(input)", fontsize=7)
    ax0.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    ax0.set_ylabel(f"num_blob={res['num_blob']}\nR²={res['r2_test']:.2f}", fontsize=8)

    # Columns 1..n_top: filter response maps
    for col, fi in enumerate(top_idx):
        kernel = torch.from_numpy(np.abs(w[fi:fi+1])).float()
        response = F.conv2d(density_t, kernel, padding=pad).squeeze().numpy()

        ax = axes[row, col + 1]
        ax.imshow(response, cmap="magma", aspect="equal", origin="lower",
                  extent=[0, gs, 0, gs])
        overlay_trajectory_on_ax(ax, trajectory_grid, arrows=arrows if col == 0 else None)
        ax.set_title(f"f{fi}  align={aligns[fi]:.4f}", fontsize=7)
        ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

fig.suptitle(
    f"Col 0: input density  |  Cols 1–{n_top}: top-{n_top} filter response maps  "
    f"(trajectory=green, num_noise={num_noise_fixed})",
    fontsize=10,
)
plt.show()

In [ ]:
# Radial density profile: does the trajectory ring sharpen with more blob particles?
# If hypothesis holds, a local peak should emerge at the trajectory radius as num_blob increases.
fig, ax = plt.subplots(figsize=(8, 5))

colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(sweep_results)))

for res, color in zip(sweep_results, colors):
    avg_density = res["avg_density"]
    grid_size   = res["meta"]["grid_size"]

    cx, cy = grid_size / 2.0, grid_size / 2.0
    ys, xs = np.mgrid[0:grid_size, 0:grid_size]
    r = np.sqrt((xs - cx) ** 2 + (ys - cy) ** 2)

    r_bins = np.linspace(0, grid_size / 2, 20)
    r_centers = 0.5 * (r_bins[:-1] + r_bins[1:])
    profile = np.array([
        avg_density[(r >= r_bins[i]) & (r < r_bins[i + 1])].mean()
        if np.any((r >= r_bins[i]) & (r < r_bins[i + 1])) else np.nan
        for i in range(len(r_bins) - 1)
    ])

    ax.plot(r_centers, profile, color=color, linewidth=2,
            label=f"num_blob={res['num_blob']}  $R^2$={res['r2_test']:.2f}")

# Trajectory radius in grid coords (same for all runs)
grid_size = sweep_results[0]["meta"]["grid_size"]
center = np.array([grid_size / 2.0, grid_size / 2.0])
traj_r_grid = np.linalg.norm(sweep_results[0]["trajectory_grid"] - center, axis=1).mean()
ax.axvline(traj_r_grid, color="lime", linestyle="--", linewidth=1.5,
           label=f"trajectory radius $\\approx$ {traj_r_grid:.1f} cells")

ax.set_xlabel("Distance from grid center (cells)")
ax.set_ylabel("Mean particle density")
ax.set_title(
    f"Radial density profile vs blob count  (num_noise={num_noise_fixed})\n"
    "A local peak at the trajectory radius indicates a distinct density ring"
)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## Summary: hypothesis test results

**Hypothesis:** More blob particles grouped together → more "votes" for the same coherent pattern → CPIC compresses the trajectory structure more easily, even with many noise particles present.

---

**Finding 1 — $R^2$ vs blob count (summary plot, left)**
The linear probe's test $R^2$ increases monotonically from 0.58 at `num_blob=5` to 0.99 at `num_blob=50`, with `num_noise=80` fixed throughout. This directly supports the hypothesis: CPIC's latent representation encodes the blob's position along the trajectory increasingly well as the blob grows larger relative to the noise population.

**Finding 2 — Filter-trajectory alignment vs blob count (summary plot, right)**
The trajectory alignment score — measuring how much of each filter's response on the density map lands on the trajectory path — rises from ~0.006 to ~0.010 across the sweep. The trend is consistent with the hypothesis but the absolute values are small and the differences are subtle, which points to a more nuanced mechanism (explained below).

**Finding 3 — Filter response maps**
The top-4 trajectory-aligned filters all produce nearly identical Gaussian-blob response maps: bright at the grid center, falling off toward the edges, with the trajectory sitting at the bright-to-dark boundary. The filters are not learning ring detectors. Rather, they act as local smoothers on the occupancy grid, and the ring's location at the boundary of the density falloff is what changes with blob count.

**Finding 4 — Radial density profile (the mechanistic explanation)**
This is the clearest result. With `num_blob=5`, the radial density profile is nearly flat at the trajectory radius — the 5 blob particles contribute a ring too weak to distinguish from 80 noise particles. As `num_blob` grows, a sharp peak emerges at the trajectory radius ($\approx$ 4.7 grid cells), becoming the dominant feature of the density field at `num_blob=50`. The $R^2$ values track this peak emergence almost exactly.

---

**Synthesis**

The hypothesis is confirmed, but the mechanism is more specific than "filters learn to detect the trajectory." What actually happens:

> More blob particles → a stronger, sharper density ridge at the trajectory radius in the 2D occupancy grid → a more reliable spatial feature for CPIC to compress → higher $R^2$ for trajectory position recovery.

The improvement is driven by **input signal quality**, not filter specialization. The filters remain generic smoothers throughout; what changes is that the ring they are smoothing over becomes increasingly distinct from the noise floor. This is the physical meaning of "more votes for the same pattern": with few blob particles, their collective density signature is swamped by noise; with many, they carve out a clear ring that CPIC can track.

## **Control:** fixed blob fraction, varying absolute count

**Question:** Does $R^2$ still improve with more particles when the blob fraction (SNR) is held constant?

**Design:** Fix `blob_fraction = 0.20` (1 blob for every 4 noise), varying `num_blob` $\in$ {5, 10, 20, 30, 50} with `num_noise = num_blob × 4`. Total N scales from 25 to 250.

The `num_blob=20, num_noise=80` point is shared with Experiment 1 — a built-in sanity check.

- If $R^2$ is **flat** across blob counts: fraction alone explains the result — absolute count adds nothing once SNR is controlled.
- If $R^2$ still **increases**: absolute count matters independently of fraction.

In [ ]:
blob_fraction_fixed = 0.20  # 1 blob per 4 noise; num_blob=20/num_noise=80 overlaps Experiment 1

fixed_fraction_results = []
for nb_val in num_blob_values:
    num_noise_val = int(round(nb_val * (1 - blob_fraction_fixed) / blob_fraction_fixed))
    total = nb_val + num_noise_val
    print(f"\n--- num_blob={nb_val}, num_noise={num_noise_val}, total={total}, fraction={nb_val/total:.2f} ---")
    res = run_conv_physical_step(
        num_blob=nb_val,
        num_noise=num_noise_val,
        conv_phys_base=conv_phys_base,
        T=T,
        device=device,
        predictive_space="observation",
    )
    print(f"  R\u00b2 test={res['r2_test']:.3f}")
    fixed_fraction_results.append(res)

print("\nFixed-fraction sweep complete.")

In [ ]:
# Comparison: R-squared vs num_blob for fixed-noise vs fixed-fraction
# Fixed-fraction holds SNR constant — if R-squared is flat, fraction explains everything.
# If R-squared still rises, absolute count matters independently.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

nb_vals = [r["num_blob"] for r in sweep_results]
nb_vals_ff = [r["num_blob"] for r in fixed_fraction_results]

ax1.plot(nb_vals, [r["r2_test"] for r in sweep_results],
         "o-", color="steelblue", linewidth=2, markersize=8,
         label=f"fixed num_noise={num_noise_fixed} (fraction varies)")
ax1.plot(nb_vals_ff, [r["r2_test"] for r in fixed_fraction_results],
         "s--", color="tomato", linewidth=2, markersize=8,
         label=f"fixed fraction={blob_fraction_fixed:.0%} (total N varies)")
for x, r in zip(nb_vals, sweep_results):
    ax1.annotate(f"{r['r2_test']:.2f}", (x, r["r2_test"]),
                 textcoords="offset points", xytext=(0, 7), ha="center", fontsize=8)
for x, r in zip(nb_vals_ff, fixed_fraction_results):
    ax1.annotate(f"{r['r2_test']:.2f}", (x, r["r2_test"]),
                 textcoords="offset points", xytext=(0, -14), ha="center", fontsize=8, color="tomato")
ax1.set_ylim(0, 1.05)
ax1.set_xlabel('Number of "blob" particles (absolute count)')
ax1.set_ylabel("Test R\u00b2")
ax1.set_title("R\u00b2 vs blob count\nfixed-noise vs fixed-fraction")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Right: R-squared vs blob fraction to show where each sweep sits in fraction space
fracs_fixed_noise = [r["num_blob"] / (r["num_blob"] + r["num_noise"]) for r in sweep_results]
fracs_fixed_frac  = [r["num_blob"] / (r["num_blob"] + r["num_noise"]) for r in fixed_fraction_results]

ax2.plot(fracs_fixed_noise, [r["r2_test"] for r in sweep_results],
         "o-", color="steelblue", linewidth=2, markersize=8,
         label=f"fixed num_noise={num_noise_fixed}")
ax2.plot(fracs_fixed_frac, [r["r2_test"] for r in fixed_fraction_results],
         "s", color="tomato", linewidth=0, markersize=10,
         label=f"fixed fraction={blob_fraction_fixed:.0%}")
ax2.axvline(blob_fraction_fixed, color="tomato", linestyle=":", linewidth=1.5,
            label=f"fraction = {blob_fraction_fixed:.0%}")
ax2.set_xlabel("Blob fraction  (num_blob / total_N)")
ax2.set_ylabel("Test R\u00b2")
ax2.set_title("R\u00b2 vs blob fraction\n(fixed-fraction points sit on a vertical line)")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

fig.suptitle("Does absolute count matter once fraction is controlled?", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Radial density profiles: fixed-fraction sweep
# Total N scales with num_blob, but the ring/background ratio stays constant.
# Any change in ring prominence is due to absolute count, not fraction.
fig, ax = plt.subplots(figsize=(8, 5))

colors = plt.cm.plasma(np.linspace(0.15, 0.85, len(fixed_fraction_results)))

for res, color in zip(fixed_fraction_results, colors):
    avg_density = res["avg_density"]
    grid_size   = res["meta"]["grid_size"]

    cx, cy = grid_size / 2.0, grid_size / 2.0
    ys, xs = np.mgrid[0:grid_size, 0:grid_size]
    r = np.sqrt((xs - cx) ** 2 + (ys - cy) ** 2)

    r_bins = np.linspace(0, grid_size / 2, 20)
    r_centers = 0.5 * (r_bins[:-1] + r_bins[1:])
    profile = np.array([
        avg_density[(r >= r_bins[i]) & (r < r_bins[i + 1])].mean()
        if np.any((r >= r_bins[i]) & (r < r_bins[i + 1])) else np.nan
        for i in range(len(r_bins) - 1)
    ])
    total = res["num_blob"] + res["num_noise"]
    ax.plot(r_centers, profile, color=color, linewidth=2,
            label=f"num_blob={res['num_blob']}  total={total}  R²={res['r2_test']:.2f}")

grid_size = fixed_fraction_results[0]["meta"]["grid_size"]
center = np.array([grid_size / 2.0, grid_size / 2.0])
traj_r_grid = np.linalg.norm(fixed_fraction_results[0]["trajectory_grid"] - center, axis=1).mean()
ax.axvline(traj_r_grid, color="lime", linestyle="--", linewidth=1.5,
           label=f"trajectory radius ≈ {traj_r_grid:.1f} cells")

ax.set_xlabel("Distance from grid center (cells)")
ax.set_ylabel("Mean particle density")
ax.set_title(
    f"Radial density profile — fixed fraction={blob_fraction_fixed:.0%}\n"
    "Ring/background ratio is constant; total density scales with num_blob"
)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## FeatureMaskMLP encoder: blob-count sweep

**Motivation:** `ConvPhysical` operates on a binned occupancy grid, so the absolute-count effect
we found is partly a grid-sampling artifact. `FeatureMaskMLP` operates directly in particle coordinate
space — each feature (particle x or y) gets a learned Bernoulli gate. This gives a direct,
artifact-free readout of which particles CPIC considers predictive.

**Design:** Same sweep as Experiment 1 (`num_noise=80` fixed, `num_blob` ∈ {5, 10, 20, 30, 50}).
Encoder: **uniform init** (all gate logits = 0, sigmoid = 0.5) + learnable 1-layer MLP, **observation predictive space**.
All gates start at equal probability — no prior knowledge injected. PI scores are computed and
overlaid as a reference line to show what the gates *should* converge to.
Trained for 200 epochs (early stop patience 50).

**Key metric — blob selection rate:** of the features whose gate probability exceeds `mask_eval_threshold`,
what fraction belong to blob particles? If CPIC correctly identifies the coherent structure,
this should exceed chance (blob feature fraction) and increase with `num_blob`.

In [ ]:
mask_eval_threshold = 0.6

mask_mlp_params = {
    "activation": "relu",
    "deterministic": False,
    "encoder_type": "mask_mlp",
    "linear_encoder": False,
    "mask_init": "uniform",
    "mask_learnable": True,
    "mask_strategy": "uniform_init_learned",
    "n_layers": 1,
}


def estimate_feature_pi_scores(data, t_split, *, max_lag=5, lag_agg="mean"):
    """Feature-wise PI proxy: lagged autocorrelation on the train split, normalised to [0,1]."""
    train = data[:t_split]
    eps = 1e-8
    lag_scores = []
    for lag in range(1, max(1, int(max_lag)) + 1):
        scores = np.zeros(train.shape[1], dtype=np.float32)
        for feat in range(train.shape[1]):
            a, b = train[:-lag, feat], train[lag:, feat]
            if np.std(a) < eps or np.std(b) < eps:
                continue
            corr = np.corrcoef(a, b)[0, 1]
            scores[feat] = 0.0 if np.isnan(corr) else abs(float(corr))
        lag_scores.append(scores)
    arr = np.stack(lag_scores, axis=0)
    result = np.max(arr, axis=0) if lag_agg == "max" else np.mean(arr, axis=0)
    max_val = float(np.max(result))
    return np.ones_like(result, dtype=np.float32) if max_val <= eps else (result / max_val).astype(np.float32)


def get_blob_mask_stats(model, particle_labels, particle_order, threshold=0.5):
    """Active fraction, blob selection rate, and mask probabilities."""
    enc = model.encoder
    if getattr(enc, "encoder_type", None) != "mask_mlp":
        return float("nan"), float("nan"), np.array([]), np.array([])

    probs = torch.sigmoid(enc._mean.mask_logits).detach().cpu().numpy()
    active_idx = np.where(probs >= threshold)[0]
    active_frac = len(active_idx) / len(probs)

    sorted_labels = particle_labels[particle_order]
    feat_labels = np.repeat(sorted_labels, 2)  # 2 features per particle: x, y

    blob_sel_rate = (
        float(np.sum(feat_labels[active_idx] == 1) / len(active_idx))
        if len(active_idx) > 0 else 0.0
    )
    return active_frac, blob_sel_rate, probs, feat_labels


def run_mask_mlp_step(num_blob, num_noise, mask_mlp_params, T, device,
                      t_max=2000, seed=42, predictive_space="observation"):
    """One sweep step: uniform init (logits=0), train mask_mlp, probe, return results."""
    d, gt, porder, plabels, smean, sstd, pos = generate_particle_process_timeseries(
        t_max=t_max, num_blob=num_blob, num_noise=num_noise,
        trajectory="circle", orbit_radius=3.0,
        omega=0.05, sigma_blob=0.1,
        noise_ar_coeff=0.8, spatial_bounds=10.0, seed=seed,
    )
    n_feat = d.shape[1]
    t_split = int(0.7 * len(d))
    train_ds = PastFutureDataset([d[:t_split]], window_size=T)

    pi_scores = estimate_feature_pi_scores(d, t_split)  # reference only, not used for init
    params = dict(mask_mlp_params)

    cpic = CPIC(ydim=2, xdim=n_feat, T=T, encoder_params=params,
                hidden_dim=64, beta=1e-3, device=device, predictive_space=predictive_space)
    cpic.to(device)
    loss_, I_c_, I_p_ = cpic.fit(X=train_ds, epochs=200, batch_size=256, lr=2e-4, early_stop=50)

    z_tr, gt_tr = build_past_windows_and_gt(d, gt, T, cpic, t_min=T, t_max=t_split)
    z_te, gt_te = build_past_windows_and_gt(d, gt, T, cpic, t_min=t_split, t_max=len(d))
    reg = LinearRegression().fit(z_tr, gt_tr)
    r2_test = r2_score(gt_te, reg.predict(z_te))

    active_frac, blob_sel_rate, mask_probs, feat_labels = get_blob_mask_stats(
        cpic, plabels, porder, threshold=mask_eval_threshold,
    )

    result = dict(
        num_blob=num_blob, num_noise=num_noise,
        r2_test=r2_test,
        active_frac=active_frac,
        blob_sel_rate=blob_sel_rate,
        mask_probs=mask_probs.copy(),
        feat_labels=feat_labels.copy(),
        pi_scores=pi_scores.copy(),
    )
    cleanup_model_resources(cpic, loss_, I_c_, I_p_)
    return result

In [ ]:
mask_mlp_results = []
for nb_val in num_blob_values:
    print(f"\n--- num_blob={nb_val}, num_noise={num_noise_fixed} ---")
    res = run_mask_mlp_step(
        num_blob=nb_val,
        num_noise=num_noise_fixed,
        mask_mlp_params=mask_mlp_params,
        T=T,
        device=device,
    )
    print(f"  R\u00b2={res['r2_test']:.3f}  active={res['active_frac']:.1%}  "
          f"blob selection={res['blob_sel_rate']:.1%}")
    mask_mlp_results.append(res)

print("\nMask sweep complete.")

In [ ]:
# Summary: R-squared and blob selection rate vs num_blob
nb_vals       = [r["num_blob"]      for r in mask_mlp_results]
r2_tests      = [r["r2_test"]       for r in mask_mlp_results]
blob_sel_rates= [r["blob_sel_rate"] for r in mask_mlp_results]
active_fracs  = [r["active_frac"]   for r in mask_mlp_results]

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))

# R-squared: FeatureMaskMLP vs ConvPhysical
ax1.plot([r["num_blob"] for r in sweep_results], [r["r2_test"] for r in sweep_results],
         "o-", color="steelblue", linewidth=2, markersize=7, label="ConvPhysical (obs)")
ax1.plot(nb_vals, r2_tests,
         "s--", color="seagreen", linewidth=2, markersize=7, label="FeatureMaskMLP (obs, uniform init)")
for x, y in zip(nb_vals, r2_tests):
    ax1.annotate(f"{y:.2f}", (x, y), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=8, color="seagreen")
ax1.set_ylim(0, 1.05)
ax1.set_xlabel('Number of "blob" particles'); ax1.set_ylabel("Test R²")
ax1.set_title("R² vs blob count"); ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)

# Blob selection rate
ax2.plot(nb_vals, blob_sel_rates, "s-", color="seagreen", linewidth=2, markersize=8)
for x, y in zip(nb_vals, blob_sel_rates):
    ax2.annotate(f"{y:.0%}", (x, y), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=9)
chance = [r["num_blob"] * 2 / ((r["num_blob"] + r["num_noise"]) * 2) for r in mask_mlp_results]
ax2.plot(nb_vals, chance, "k--", linewidth=1.2, label="chance (blob feature fraction)")
ax2.set_xlabel('Number of "blob" particles'); ax2.set_ylabel("Blob selection rate")
ax2.set_title("Blob feature selection rate\n(fraction of active gates on blob features)")
ax2.set_ylim(0, 1.09); ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

# Active fraction
ax3.plot(nb_vals, active_fracs, "^-", color="darkorange", linewidth=2, markersize=8)
for x, y in zip(nb_vals, active_fracs):
    ax3.annotate(f"{y:.0%}", (x, y), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=9)
ax3.set_xlabel('Number of "blob" particles'); ax3.set_ylabel("Active feature fraction")
ax3.set_title(f"Fraction of features kept active\n(gate prob > {mask_eval_threshold})")
ax3.set_ylim(0, 1); ax3.grid(True, alpha=0.3)

fig.suptitle(f"FeatureMaskMLP blob-count sweep — uniform init, obs space  (num_noise={num_noise_fixed})", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Feature mask grid: one row per num_blob
# Bar height = gate probability; blue = blob feature, orange = noise feature.
# Green line = PI scores (reference only, not used for init); black dashed = active threshold.
fig, axes = plt.subplots(len(mask_mlp_results), 1,
                          figsize=(14, 2.8 * len(mask_mlp_results)),
                          constrained_layout=True)
if len(mask_mlp_results) == 1:
    axes = [axes]

for ax, res in zip(axes, mask_mlp_results):
    probs       = res["mask_probs"]
    feat_labels = res["feat_labels"]
    pi_scores   = res["pi_scores"]
    x = np.arange(len(probs))
    colors = np.where(feat_labels == 1, "tab:blue", "tab:orange")
    ax.bar(x, probs, color=colors, alpha=0.85, width=1.0)
    ax.plot(x, pi_scores, color="limegreen", linewidth=1.2, alpha=0.8, label="PI scores (reference)")
    ax.axhline(mask_eval_threshold, color="k", linestyle="--", linewidth=1,
               label=f"threshold={mask_eval_threshold}")
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel("Gate prob.")
    ax.set_title(
        f"num_blob={res['num_blob']}  "
        f"R²={res['r2_test']:.2f}  "
        f"active={res['active_frac']:.0%}  "
        f"blob selection={res['blob_sel_rate']:.0%}",
        fontsize=9,
    )
    ax.legend(loc="upper right", fontsize=7)

axes[-1].set_xlabel(
    "Feature index (2 per x-sorted particle: x, y)  "
    "[blue = blob, orange = noise]"
)
fig.suptitle(
    f"FeatureMaskMLP learned gates (uniform init, learnable) — blob-count sweep  (num_noise={num_noise_fixed})\n"
    "Gates start at 0.5 for all features — convergence toward PI scores = CPIC is learning",
    fontsize=10,
)
plt.show()

## Filter response at trajectory phases

**Question:** Does the top filter's response track the blob's position as it moves along the trajectory, or is the response map static regardless of where the blob is?

**Design:** Re-generate the best-result timeseries (num_blob=50, seed=42). Pick 4 timesteps at trajectory phases 0°, 90°, 180°, 270°. For each: show the single-timestep density grid (input) and the top filter's response on that grid. The cyan dot marks the blob centroid at that timestep.

If the filter is trajectory-aware, the bright region in the response should shift to follow the blob. If the response is static, the filter is not tracking dynamics — it is responding to the fixed background density structure.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Regenerate data for the best run (largest num_blob) using same seed
best_res = sweep_results[-1]
nb_best  = best_res["num_blob"]
d_best, _, _, _, _, _, _ = generate_particle_process_timeseries(
    t_max=2000, num_blob=nb_best, num_noise=num_noise_fixed,
    trajectory="circle", orbit_radius=3.0,
    omega=0.05, sigma_blob=0.1,
    noise_ar_coeff=0.8, spatial_bounds=10.0, seed=42,
)

gs      = best_res["meta"]["grid_size"]
spb     = best_res["meta"]["spatial_bounds"]
w       = best_res["filters"]
aligns  = best_res["align_scores"]
top_fi  = int(np.argmax(aligns))
trajectory_grid_best = best_res["trajectory_grid"]  # (t_max, 2) blob centroid path in grid coords

K = w.shape[2]; pad_k = K // 2
kernel = torch.from_numpy(np.abs(w[top_fi:top_fi+1])).float()

def single_timestep_density(data_t, grid_size, spatial_bounds):
    N = len(data_t) // 2
    scale = grid_size / (2.0 * spatial_bounds)
    density = np.zeros((grid_size, grid_size), dtype=np.float32)
    coords = data_t.reshape(N, 2)
    xs = np.clip(((coords[:, 0] + spatial_bounds) * scale).astype(int), 0, grid_size - 1)
    ys = np.clip(((coords[:, 1] + spatial_bounds) * scale).astype(int), 0, grid_size - 1)
    np.add.at(density, (ys, xs), 1.0)
    return density

t_offset = 200

# Precompute colorscale limits across one full period for consistent colormap
density_max = response_max = 0.0
for t in range(t_offset, t_offset + period_steps):
    d_t = single_timestep_density(d_best[t], gs, spb)
    r_t = F.conv2d(torch.from_numpy(d_t).float().unsqueeze(0).unsqueeze(0),
                   kernel, padding=pad_k).squeeze().numpy()
    density_max = max(density_max, float(d_t.max()))
    response_max = max(response_max, float(r_t.max()))

# Build figure with two rows: input density (top) and filter response (bottom)
fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(4.5, 8), constrained_layout=True)

d0 = single_timestep_density(d_best[t_offset], gs, spb)
r0 = F.conv2d(torch.from_numpy(d0).float().unsqueeze(0).unsqueeze(0),
              kernel, padding=pad_k).squeeze().numpy()

im_top = ax_top.imshow(d0, cmap="magma", origin="lower", aspect="equal",
                        extent=[0, gs, 0, gs], vmin=0, vmax=density_max)
im_bot = ax_bot.imshow(r0, cmap="magma", origin="lower", aspect="equal",
                        extent=[0, gs, 0, gs], vmin=0, vmax=response_max)

for ax in (ax_top, ax_bot):
    ax.plot(trajectory_grid_best[:, 0], trajectory_grid_best[:, 1],
            color="lime", linewidth=0.8, alpha=0.6)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

dot_top = ax_top.scatter([], [], color="cyan", s=60, zorder=7)
dot_bot = ax_bot.scatter([], [], color="cyan", s=60, zorder=7)
ax_top.set_ylabel("Input density", fontsize=9)
ax_bot.set_ylabel(f"Filter f{top_fi} response", fontsize=9)
title = fig.suptitle("", fontsize=9)

def update(frame):
    t = t_offset + frame
    density = single_timestep_density(d_best[t], gs, spb)
    response = F.conv2d(torch.from_numpy(density).float().unsqueeze(0).unsqueeze(0),
                        kernel, padding=pad_k).squeeze().numpy()
    im_top.set_data(density)
    im_bot.set_data(response)
    blob_xy = trajectory_grid_best[t]
    dot_top.set_offsets([[blob_xy[0], blob_xy[1]]])
    dot_bot.set_offsets([[blob_xy[0], blob_xy[1]]])
    phase_deg = 360.0 * frame / period_steps
    title.set_text(
        f"f{top_fi} response tracking blob trajectory — num_blob={nb_best}\n"
        f"t={t}  phase={phase_deg:.0f}°  (cyan = blob centroid)"
    )
    return im_top, im_bot, dot_top, dot_bot, title

anim = FuncAnimation(fig, update, frames=period_steps, interval=80, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())

In [ ]:
# Save the trajectory-tracking animation as a GIF for the slide deck / web.
# interval=80 ms in FuncAnimation -> ~12 fps.
from matplotlib.animation import PillowWriter

gif_path = "res/convphysical_filter_response_trajectory.gif"
anim.save(gif_path, writer=PillowWriter(fps=12), dpi=110)
print("wrote", gif_path)

## Trajectory confound test: ellipse vs circle

**Motivation:** In all previous experiments the blob traces a symmetric circle of radius 3.0, which sits exactly at the edge of the 20×20 occupancy grid (encoder `spatial_bounds=3.0`). The AR(1) noise is mean-reverting to the origin, so the grid's center is always high-density noise and the orbit sits at the low-density periphery. A generic center-peaked Gaussian smoother can distinguish these regions without learning anything about the orbit's geometry — this is the center-density confound.

**Test:** Replace the circle with an ellipse (`semi_major=2.5`, `semi_minor=1.5`). The blob now traces a non-symmetric path: along the x-axis it reaches ±2.5 (near the periphery) but along the y-axis it only reaches ±1.5 (well within the noisy center region). If CPIC's R² degrades significantly on the ellipse, the circle result was exploiting the center-periphery contrast, not the orbit structure. If R² holds up, CPIC is genuinely tracking the coherent trajectory regardless of where it sits in the density field.

**Design:** `num_blob=30`, `num_noise=80`, `n_layers=0`, same seed. Compare circle vs ellipse side-by-side.

In [ ]:
ellipse_num_blob  = 30
ellipse_num_noise = 80
ellipse_semi_major = 2.5
ellipse_semi_minor = 1.5

print("--- circle (radius=3.0) ---")
res_circle = run_conv_physical_step(
    num_blob=ellipse_num_blob, num_noise=ellipse_num_noise,
    conv_phys_base=conv_phys_base, T=T, device=device,
    trajectory="circle", seed=42,
)
print(f"  R² test={res_circle['r2_test']:.3f}  max align={res_circle['max_align']:.4f}")

print(f"\n--- ellipse (a={ellipse_semi_major}, b={ellipse_semi_minor}) ---")
res_ellipse = run_conv_physical_step(
    num_blob=ellipse_num_blob, num_noise=ellipse_num_noise,
    conv_phys_base=conv_phys_base, T=T, device=device,
    trajectory="ellipse", semi_major=ellipse_semi_major, semi_minor=ellipse_semi_minor, seed=42,
)
print(f"  R² test={res_ellipse['r2_test']:.3f}  max align={res_ellipse['max_align']:.4f}")

print("\nEllipse confound test complete.")

In [ ]:
# Side-by-side: avg density + top-4 filter responses for circle vs ellipse
n_top  = 4
n_cols = n_top + 1
traj_results = [res_circle, res_ellipse]
traj_labels  = [f"circle  r=3.0", f"ellipse  a={ellipse_semi_major} b={ellipse_semi_minor}"]

fig, axes = plt.subplots(2, n_cols, figsize=(n_cols * 2.6, 2 * 2.6), constrained_layout=True)

for row, (res, label) in enumerate(zip(traj_results, traj_labels)):
    w               = res["filters"]
    trajectory_grid = res["trajectory_grid"]
    avg_density     = res["avg_density"]
    aligns          = res["align_scores"]
    top_idx         = np.argsort(aligns)[::-1][:n_top]
    arrows          = compute_tangential_arrows(trajectory_grid)
    gs              = res["meta"]["grid_size"]

    density_t = torch.from_numpy(avg_density).float().unsqueeze(0).unsqueeze(0)
    K = w.shape[2]; pad = K // 2

    ax0 = axes[row, 0]
    ax0.imshow(avg_density, cmap="magma", aspect="equal", origin="lower", extent=[0, gs, 0, gs])
    overlay_trajectory_on_ax(ax0, trajectory_grid, arrows=arrows)
    ax0.set_title("avg density\n(input)", fontsize=7)
    ax0.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    ax0.set_ylabel(f"{label}\nR²={res['r2_test']:.3f}", fontsize=8)

    for col, fi in enumerate(top_idx):
        kernel   = torch.from_numpy(np.abs(w[fi:fi+1])).float()
        response = F.conv2d(density_t, kernel, padding=pad).squeeze().numpy()
        ax = axes[row, col + 1]
        ax.imshow(response, cmap="magma", aspect="equal", origin="lower", extent=[0, gs, 0, gs])
        overlay_trajectory_on_ax(ax, trajectory_grid, arrows=arrows if col == 0 else None)
        ax.set_title(f"f{fi}  align={aligns[fi]:.4f}", fontsize=7)
        ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

fig.suptitle(
    f"Trajectory confound test — circle vs ellipse\n"
    f"(num_blob={ellipse_num_blob}, num_noise={ellipse_num_noise}, n_layers=0, seed=42)",
    fontsize=10,
)
plt.show()